# 04 — Pelatihan dan Evaluasi Model

**Seluruh metrik pada notebook ini dihitung di atas DATA SINTETIS.**
Angkanya membuktikan pipeline berjalan benar, bukan mengukur kemampuan
deteksi di PLTU Jeranjang Unit 1.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from backend.app.core.config import get_settings
settings = get_settings()
print("Akar proyek:", settings.paths.root)
print("Mode deployment:", settings.deployment_mode)

In [ ]:
from backend.app.core.constants import SYNTHETIC_METRIC_WARNING
print(SYNTHETIC_METRIC_WARNING)

## Pembagian waktu

In [ ]:
from backend.app.models.train import build_split_plan

plan = build_split_plan(settings)
print("Latih    :", plan.train_years)
print("Validasi :", plan.validation_years)
print("Uji      :", plan.test_years)
print()
print("Embargo  :", settings.model["split"]["embargo_hours"], "jam")
print("Batas latih berakhir   :", plan.train_end)
print("Validasi dimulai       :", plan.validation_start)
print("Validasi berakhir      :", plan.validation_end)
print("Uji dimulai            :", plan.test_start)

## Pelatihan

Berjalan sekitar dua puluh menit. Lewati bila artefaknya sudah ada dan
langsung baca tabel evaluasi di bawah.

In [ ]:
# from backend.app.models.train import run
# outcome = run(settings)

## Hasil evaluasi

In [ ]:
table = pd.read_csv(settings.paths.reports / "model_evaluation.csv")
table.loc[table["dataset"] == "test"].sort_values(["horizon", "pr_auc"], ascending=[True, False])

## Mengapa ROC-AUC menyesatkan di sini

Dengan prevalensi positif di bawah satu persen, model yang selalu
menjawab "tidak ada risiko" sudah benar hampir sepanjang waktu.

In [ ]:
test = table.loc[table["dataset"] == "test"]
test[["model", "horizon", "positive_rate", "roc_auc", "pr_auc", "precision", "recall"]]

## Metrik per event — pertanyaan yang sebenarnya ditanyakan operator

In [ ]:
test[["model", "horizon", "event_count", "events_detected", "event_detection_rate",
      "false_alarms_per_day", "median_warning_horizon_minutes",
      "average_warning_horizon_minutes"]]

## Kalibrasi

In [ ]:
test[["model", "horizon", "brier_score", "calibration_error"]].sort_values("calibration_error")

## Registri model

In [ ]:
import json

payload = json.loads((settings.paths.models / "model_registry.json").read_text(encoding="utf-8"))
print("Sumber data:", payload["metadata"]["data_source"])
print()
for entry in payload["models"]:
    print(f"{entry['name']:22} {entry['horizon']:20} ambang {entry['threshold']:.5f}")

## Fitur paling berpengaruh menurut SHAP

In [ ]:
import joblib
from backend.app.explainability.shap_explainer import ShapExplainer
from backend.app.models.train import Dataset

horizon = settings.model["primary_horizon"]
artifact = joblib.load(settings.paths.models / f"xgboost_{horizon}.joblib")

sample = pd.read_parquet(settings.paths.processed / "dataset_test.parquet").head(2000)
explainer = ShapExplainer(artifact["model"], artifact["columns"]).fit()
explainer.global_importance(sample, top_n=20)